In [1]:
from IPython.core.interactiveshell import InteractiveShell

InteractiveShell.ast_node_interactivity = "all"

In [2]:
import hopsworks

In [7]:
import os
from dotenv import dotenv_values
ENV = dotenv_values("./work/AIEngineering/machine_learning/feature_store/hopsworks/.env")

In [8]:
project = hopsworks.login(
    project='my_first_project10',  
    host="eu-west.cloud.hopsworks.ai",
    port=443,
    api_key_value=ENV['api_key']
)

2026-09-03 18:09:08,161 INFO: Closing external client and cleaning up certificates.
2026-09-03 18:09:08,169 INFO: Connection closed.
2026-09-03 18:09:08,171 INFO: Initializing external client
2026-09-03 18:09:08,173 INFO: Base URL: https://eu-west.cloud.hopsworks.ai:443
2026-09-03 18:09:10,272 INFO: Python Engine initialized.



Logged in to project, explore it here https://eu-west.cloud.hopsworks.ai:443/p/42213


In [ ]:
fs = project.get_feature_store()

In [ ]:
fg_comportamento_financeiro = fs.get_feature_group(
    name="pessoa_comportamento_financeiro",
    version=2
)


fg_comportamento_identitario = fs.get_feature_group(
    name="pessoa_comportamento_identitario",
    version=2
)

fg_financeiro_plus = fs.get_feature_group(
    name="pessoa_comportamento_financeiro_plus",
    version=2
)

In [11]:
fg_financeiro_plus.read()

Finished: Reading data from Hopsworks, using Hopsworks Feature Query Service (1.87s) 


,pessoa_id,gastos_100d,inadimplente,timestamp
0,1001,1200.0,0,2025-03
1,1002,8500.0,0,2026-01
2,1003,400.0,1,2022-01
3,1001,2000.0,0,2025-05


In [ ]:
fg_financeiro_plus.read().filter('customer_id', 1001)

ConnectionError: ('Connection aborted.', RemoteDisconnected('Remote end closed connection without response'))

In [13]:
fg_financeiro_plus.read().filter('customer_id', 1001)

2026-09-03 18:14:31,788 ERROR: Flight returned unavailable error, with message: Socket closed. gRPC client debug context: UNKNOWN:Error received from peer ipv4:57.130.64.132:5005 {grpc_status:14, grpc_message:"Socket closed"}. Client context: IOError: Server never sent a data message. Detail: Internal
Traceback (most recent call last):
  File "/opt/conda/lib/python3.11/site-packages/hsfs/core/arrow_flight_client.py", line 433, in afs_error_handler_wrapper
    return func(instance, *args, **kw)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/opt/conda/lib/python3.11/site-packages/hsfs/core/arrow_flight_client.py", line 505, in _read_query
    return self._get_dataset(
           ^^^^^^^^^^^^^^^^^^
  File "/opt/conda/lib/python3.11/site-packages/retrying.py", line 55, in wrapped_f
    return Retrying(*dargs, **dkw).call(f, *args, **kw)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/opt/conda/lib/python3.11/site-packages/retrying.py", line 289, in call
    raise attempt.

Error: Reading data from Hopsworks, using Hopsworks Feature Query Service           


FeatureStoreException: Could not read data using Hopsworks Query Service.

In [22]:
entity = fg_financeiro_plus.filter(fg_financeiro_plus.pessoa_id == 1001).read()

Finished: Reading data from Hopsworks, using Hopsworks Feature Query Service (1.95s) 


In [23]:
type(entity)

pandas.core.frame.DataFrame

In [ ]:
fg_financeiro_plus.read("20250501")

#O Hopsworks tem dois "motores" (engines) de execução no Python SDK
#Engine Python (padrão em notebooks locais/Jupyter sem Spark): Usa o Arrow Flight Engine para consultar dados de forma rápida, mas não possui suporte a consultas temporais/time-travel (wallclock_time="20250501").
#Engine Python (padrão em notebooks locais/Jupyter sem Spark): Usa o Arrow Flight Engine para consultar dados de forma rápida, mas não possui suporte a consultas temporais/time-travel (wallclock_time="20250501").
#Quando você passou "20250501" como primeiro argumento para o .read(), o SDK interpretou esse valor como o argumento wallclock_time. Como seu ambiente está usando a engine Python, o Hopsworks bloqueou a execução.


FeatureStoreException: Python environments does not support incremental queries. Read feature group without timestamp to retrieve latest snapshot or switch to environment with Spark Engine.

In [19]:
fg_financeiro_plus.read(online=True)

,pessoa_id,gastos_100d,inadimplente,timestamp
0,1003,400.0,1,2022-01
1,1002,8500.0,0,2026-01
2,1001,2000.0,0,2025-05


In [20]:
fg_financeiro_plus.read(online=False)

Finished: Reading data from Hopsworks, using Hopsworks Feature Query Service (74.43s) 


,pessoa_id,gastos_100d,inadimplente,timestamp
0,1001,1200.0,0,2025-03
1,1002,8500.0,0,2026-01
2,1003,400.0,1,2022-01
3,1001,2000.0,0,2025-05


In [21]:
fg_financeiro_plus.read()

Finished: Reading data from Hopsworks, using Hopsworks Feature Query Service (40.42s) 


,pessoa_id,gastos_100d,inadimplente,timestamp
0,1001,1200.0,0,2025-03
1,1002,8500.0,0,2026-01
2,1003,400.0,1,2022-01
3,1001,2000.0,0,2025-05
